In [1]:
from typing import Optional
from datetime import datetime
from pydantic import BaseModel, Field
from openai import OpenAI
import os
import logging

In [2]:
# Set up logging configuration
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
model = "gpt-5-nano"

In [3]:
class EventExtraction(BaseModel):
    """First LLM call: Extract basic event information"""

    description: str = Field(description="Raw description of the event")
    is_calendar_event: bool = Field(
        description="Whether this text describes a calendar event"
    )
    confidence_score: float = Field(description="Confidence score between 0 and 1")


class EventDetails(BaseModel):
    """Second LLM call: Parse specific event details"""

    name: str = Field(description="Name of the event")
    date: str = Field(
        description="Date and time of the event. Use ISO 8601 to format this value."
    )
    duration_minutes: int = Field(description="Expected duration in minutes")
    participants: list[str] = Field(description="List of participants")


class EventConfirmation(BaseModel):
    """Third LLM call: Generate confirmation message"""

    confirmation_message: str = Field(
        description="Natural language confirmation message"
    )
    calendar_link: Optional[str] = Field(
        description="Generated calendar link if applicable"
    )

In [4]:
def extract_event_info(user_input: str) -> EventExtraction:
    """First LLM call to determine if input is a calendar event"""
    logger.info("Starting event extraction analysis")
    logger.debug(f"Input text: {user_input}")

    today = datetime.now()
    date_context = f"Today is {today.strftime('%A, %B %d, %Y')}."

    completion = client.beta.chat.completions.parse(
        model=model,
        messages=[
            {
                "role": "system",
                "content": f"{date_context} Analyze if the text describes a calendar event.",
            },
            {"role": "user", "content": user_input},
        ],
        response_format=EventExtraction,
    )
    result = completion.choices[0].message.parsed
    print("extract_event_info_dump\n")
    print(result.model_dump())
    logger.info(
        f"Extraction complete - Is calendar event: {result.is_calendar_event}, Confidence: {result.confidence_score:.2f}"
    )
    return result

In [5]:
def parse_event_details(description: str) -> EventDetails:
    """Second LLM call to extract specific event details"""
    logger.info("Starting event details parsing")

    today = datetime.now()
    date_context = f"Today is {today.strftime('%A, %B %d, %Y')}."

    completion = client.beta.chat.completions.parse(
        model=model,
        messages=[
            {
                "role": "system",
                "content": f"{date_context} Extract detailed event information. When dates reference 'next Tuesday' or similar relative dates, use this current date as reference.",
            },
            {"role": "user", "content": description},
        ],
        response_format=EventDetails,
    )
    result = completion.choices[0].message.parsed
    print("parse_event_details_dump\n")
    print(result.model_dump())
    logger.info(
        f"Parsed event details - Name: {result.name}, Date: {result.date}, Duration: {result.duration_minutes}min"
    )
    logger.debug(f"Participants: {', '.join(result.participants)}")
    return result

In [6]:
def generate_confirmation(event_details: EventDetails) -> EventConfirmation:
    """Third LLM call to generate a confirmation message"""
    logger.info("Generating confirmation message")

    completion = client.beta.chat.completions.parse(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "Generate a natural confirmation message for the event. Sign of with your name; Susie",
            },
            {"role": "user", "content": str(event_details.model_dump())},
        ],
        response_format=EventConfirmation,
    )
    result = completion.choices[0].message.parsed
    print("generate_confirmation_dump\n")
    print(result.model_dump())
    logger.info("Confirmation message generated successfully")
    return result

In [7]:
def process_calendar_request(user_input: str) -> Optional[EventConfirmation]:
    """Main function implementing the prompt chain with gate check"""
    logger.info("Processing calendar request")
    logger.debug(f"Raw input: {user_input}")

    # First LLM call: Extract basic info
    initial_extraction = extract_event_info(user_input)

    # Gate check: Verify if it's a calendar event with sufficient confidence
    if (
        not initial_extraction.is_calendar_event
        or initial_extraction.confidence_score < 0.7
    ):
        logger.warning(
            f"Gate check failed - is_calendar_event: {initial_extraction.is_calendar_event}, confidence: {initial_extraction.confidence_score:.2f}"
        )
        return None

    logger.info("Gate check passed, proceeding with event processing")

    # Second LLM call: Get detailed event information
    event_details = parse_event_details(initial_extraction.description)

    # Third LLM call: Generate confirmation
    confirmation = generate_confirmation(event_details)

    logger.info("Calendar request processing completed successfully")
    return confirmation

In [8]:
user_input = "Let's schedule a 1h team meeting next Tuesday at 2pm with Alice and Bob to discuss the project roadmap."

result = process_calendar_request(user_input)
if result:
    print(f"Confirmation: {result.confirmation_message}")
    if result.calendar_link:
        print(f"Calendar Link: {result.calendar_link}")
else:
    print("This doesn't appear to be a calendar event request.")

2025-10-08 20:15:01 - INFO - Processing calendar request
2025-10-08 20:15:01 - INFO - Starting event extraction analysis
2025-10-08 20:15:16 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-08 20:15:16 - INFO - Extraction complete - Is calendar event: True, Confidence: 0.88
2025-10-08 20:15:16 - INFO - Gate check passed, proceeding with event processing
2025-10-08 20:15:16 - INFO - Starting event details parsing


extract_event_info_dump

{'description': '1h team meeting next Tuesday at 2pm with Alice and Bob to discuss the project roadmap.', 'is_calendar_event': True, 'confidence_score': 0.88}


2025-10-08 20:15:44 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-08 20:15:44 - INFO - Parsed event details - Name: Team meeting - Project roadmap discussion, Date: 2025-10-14T14:00:00, Duration: 60min
2025-10-08 20:15:44 - INFO - Generating confirmation message


parse_event_details_dump

{'name': 'Team meeting - Project roadmap discussion', 'date': '2025-10-14T14:00:00', 'duration_minutes': 60, 'participants': ['Alice', 'Bob']}


2025-10-08 20:15:59 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-08 20:15:59 - INFO - Confirmation message generated successfully
2025-10-08 20:15:59 - INFO - Calendar request processing completed successfully


generate_confirmation_dump

{'confirmation_message': "Your meeting 'Team meeting - Project roadmap discussion' is confirmed for October 14, 2025 at 2:00 PM (local time) for 60 minutes. Participants: Alice, Bob. If you need to adjust details, let me know. Best regards, Susie", 'calendar_link': None}
Confirmation: Your meeting 'Team meeting - Project roadmap discussion' is confirmed for October 14, 2025 at 2:00 PM (local time) for 60 minutes. Participants: Alice, Bob. If you need to adjust details, let me know. Best regards, Susie


In [9]:
user_input = "Can you send an email to Alice and Bob to discuss the project roadmap?"

result = process_calendar_request(user_input)
if result:
    print(f"Confirmation: {result.confirmation_message}")
    if result.calendar_link:
        print(f"Calendar Link: {result.calendar_link}")
else:
    print("This doesn't appear to be a calendar event request.")

2025-10-08 20:17:13 - INFO - Processing calendar request
2025-10-08 20:17:13 - INFO - Starting event extraction analysis
2025-10-08 20:17:19 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-08 20:17:19 - INFO - Extraction complete - Is calendar event: False, Confidence: 0.78
2025-10-08 20:17:19 - WARNING - Gate check failed - is_calendar_event: False, confidence: 0.78


extract_event_info_dump

{'description': 'Can you send an email to Alice and Bob to discuss the project roadmap?', 'is_calendar_event': False, 'confidence_score': 0.78}
This doesn't appear to be a calendar event request.
